In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "font.size": 11,
    "axes.titlesize": 13,
})

np.random.seed(42)
torch.manual_seed(42)

print("Ready.")

Ready.


---
# 1) Loss Functions

The loss function defines **what** we optimize. Choosing the right loss is as important as choosing the right architecture.
---

## 1.1 Binary Cross-Entropy (BCE)

This remarkably clean gradient is why sigmoid + BCE is a natural pairing.

In [ ]:
def binary_cross_entropy_numpy(y_pred, y_true, eps=1e-8):
    """Compute BCE from scratch.

    y_pred: (batch_num,) — predicted probabilities
    y_true: (batch_num,) — ground truth {0, 1}
    Returns: scalar loss
    """
    y_pred = np.clip(y_pred, eps, 1 - eps)
    # scalar
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


# Compare with PyTorch
y_pred_np = np.array([0.9, 0.2, 0.7, 0.1, 0.8])
y_true_np = np.array([1.0, 0.0, 1.0, 0.0, 1.0])

loss_np = binary_cross_entropy_numpy(y_pred_np, y_true_np)

y_pred_pt = torch.tensor(y_pred_np, dtype=torch.float32)
y_true_pt = torch.tensor(y_true_np, dtype=torch.float32)
loss_pt = F.binary_cross_entropy(y_pred_pt, y_true_pt).item()

print(f"NumPy BCE:   {loss_np:.8f}")
print(f"PyTorch BCE: {loss_pt:.8f}")
print(f"Match: {np.isclose(loss_np, loss_pt, atol=1e-6)}")

NumPy BCE:   0.20273662
PyTorch BCE: 0.20273665
Match: True


---
## 1.2 Categorical Cross-Entropy — Derivation

Same clean form as BCE! The softmax-CE combination has the same derivative structure as sigmoid-BCE.

In [ ]:
def softmax_numpy(z):
    """Numerically stable softmax.

    z: (num_classes, batch_num)
    Returns: (num_classes, batch_num)
    """
    z_shifted = z - z.max(axis=0, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=0, keepdims=True)


def categorical_cross_entropy_numpy(logits, y_onehot, eps=1e-8):
    """Compute categorical CE from scratch.

    logits: (num_classes, batch_num) — raw logits
    y_onehot: (num_classes, batch_num) — one-hot targets
    Returns: scalar loss
    """
    # (num_classes, batch_num)
    probs = softmax_numpy(logits)
    # scalar
    return -np.mean(np.sum(y_onehot * np.log(probs + eps), axis=0))


# Compare with PyTorch
np.random.seed(0)
num_classes, batch_num = 5, 8
# (num_classes, batch_num)
logits_np = np.random.randn(num_classes, batch_num)
# (batch_num,)
targets = np.random.randint(0, num_classes, size=batch_num)
# (num_classes, batch_num)
y_onehot = np.eye(num_classes)[targets].T

loss_np = categorical_cross_entropy_numpy(logits_np, y_onehot)

logits_pt = torch.tensor(logits_np.T, dtype=torch.float32)
targets_pt = torch.tensor(targets, dtype=torch.long)
loss_pt = F.cross_entropy(logits_pt, targets_pt).item()

print(f"NumPy CE:   {loss_np:.8f}")
print(f"PyTorch CE: {loss_pt:.8f}")
print(f"Match: {np.isclose(loss_np, loss_pt, atol=1e-5)}")

NumPy CE:   2.07017486
PyTorch CE: 2.07017493
Match: True


---
## 1.3 Negative Log-Likelihood (NLL) + LogSoftmax

Together, `NLLLoss(LogSoftmax(z))` ≡ `CrossEntropyLoss(z)`.

The decomposition avoids computing $\log(\text{softmax})$ which involves $\log(\exp(\cdot))$ — a numerically dangerous operation.

In [ ]:
def log_softmax_numpy(z):
    """Numerically stable log-softmax.

    z: (num_classes, batch_num)
    Returns: (num_classes, batch_num)
    """
    z_shifted = z - z.max(axis=0, keepdims=True)
    # (num_classes, batch_num)
    return z_shifted - np.log(np.exp(z_shifted).sum(axis=0, keepdims=True))


def nll_loss_numpy(log_probs, targets):
    """Negative log-likelihood loss.

    log_probs: (num_classes, batch_num)
    targets: (batch_num,) — integer class labels
    Returns: scalar
    """
    batch_num = targets.shape[0]
    # Pick the log-probability of the correct class for each sample
    # scalar
    return -np.mean(log_probs[targets, np.arange(batch_num)])


# Verify NLL(LogSoftmax(z)) == CE(z)
log_probs = log_softmax_numpy(logits_np)
loss_nll = nll_loss_numpy(log_probs, targets)

# PyTorch equivalent
log_probs_pt = F.log_softmax(torch.tensor(logits_np.T, dtype=torch.float32), dim=1)
loss_nll_pt = F.nll_loss(log_probs_pt, targets_pt).item()

print(f"NumPy NLL(LogSoftmax):   {loss_nll:.8f}")
print(f"PyTorch NLL(LogSoftmax): {loss_nll_pt:.8f}")
print(f"Equals CE?               {np.isclose(loss_nll, loss_np, atol=1e-6)}")

NumPy NLL(LogSoftmax):   2.07017498
PyTorch NLL(LogSoftmax): 2.07017493
Equals CE?               True


---
## 1.4 KL Divergence — Derivation, Relation to Cross-Entropy

The **Kullback-Leibler divergence** measures how one distribution $Q$ diverges from a reference distribution $P$:

$$D_{\text{KL}}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)} = \underbrace{-\sum_x P(x) \log Q(x)}_{\text{Cross-Entropy } H(P, Q)} - \underbrace{\left(-\sum_x P(x) \log P(x)\right)}_{\text{Entropy } H(P)}$$

Therefore:

$$D_{\text{KL}}(P \| Q) = H(P, Q) - H(P)$$

Since $H(P)$ is a constant w.r.t. model parameters, **minimizing CE ≡ minimizing KL divergence** between the data distribution and the model.

**Properties:**
- $D_{\text{KL}} \geq 0$ (Gibbs' inequality)
- $D_{\text{KL}} = 0 \iff P = Q$
- **Not symmetric**: $D_{\text{KL}}(P \| Q) \neq D_{\text{KL}}(Q \| P)$

In [ ]:
def kl_divergence_numpy(p, q, eps=1e-8):
    """Compute KL(P || Q) for discrete distributions.

    p, q: (num_classes,) or (num_classes, batch_num)
    Returns: scalar or (batch_num,)
    """
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q), axis=0)


# Example: two distributions
P = np.array([0.4, 0.3, 0.2, 0.1])
Q = np.array([0.25, 0.25, 0.25, 0.25])  # uniform

kl_pq = kl_divergence_numpy(P, Q)
kl_qp = kl_divergence_numpy(Q, P)

# Cross-entropy = KL + entropy
H_P = -np.sum(P * np.log(P))
H_PQ = -np.sum(P * np.log(Q))

print(f"KL(P || Q) = {kl_pq:.6f}")
print(f"KL(Q || P) = {kl_qp:.6f}  (asymmetric!)")
print(f"H(P)       = {H_P:.6f}")
print(f"H(P, Q)    = {H_PQ:.6f}")
print(f"KL(P||Q) + H(P) = {kl_pq + H_P:.6f} == H(P,Q)? {np.isclose(kl_pq + H_P, H_PQ)}")

# PyTorch verification
kl_pt = F.kl_div(
    torch.tensor(Q, dtype=torch.float32).log(),
    torch.tensor(P, dtype=torch.float32),
    reduction="sum",
).item()
print(f"\nPyTorch KL: {kl_pt:.6f} (matches NumPy: {np.isclose(kl_pq, kl_pt, atol=1e-5)})")

KL(P || Q) = 0.106440
KL(Q || P) = 0.121777  (asymmetric!)
H(P)       = 1.279854
H(P, Q)    = 1.386294
KL(P||Q) + H(P) = 1.386294 == H(P,Q)? True

PyTorch KL: 0.106440 (matches NumPy: True)


---
## 1.5 Label Smoothing — Why and How

Hard targets $y = [0, 0, 1, 0]$ push the model toward infinite logit confidence. **Label smoothing** (Szegedy et al., 2016) softens targets:

$$y_c^{\text{smooth}} = (1 - \epsilon) \cdot y_c + \frac{\epsilon}{C}$$

For the correct class: $y_c = 1 - \epsilon + \epsilon/C$. For incorrect classes: $y_c = \epsilon/C$.

**Why it helps:**
1. Prevents overconfident predictions (improves calibration)
2. Acts as a regularizer on the logit magnitudes
3. The smoothed CE loss is equivalent to: $(1-\epsilon) \cdot H(y, \hat{y}) + \epsilon \cdot H(u, \hat{y})$ where $u$ is uniform — it pulls predictions toward uniform, preventing extreme confidence.

In [ ]:
def label_smoothing(y_onehot, epsilon=0.1):
    """Apply label smoothing to one-hot targets.

    y_onehot: (num_classes, batch_num)
    Returns: (num_classes, batch_num)
    """
    num_classes = y_onehot.shape[0]
    return (1 - epsilon) * y_onehot + epsilon / num_classes


def cross_entropy_with_label_smoothing(logits, y_onehot, epsilon=0.1, eps=1e-8):
    """CE loss with label smoothing applied.

    logits: (num_classes, batch_num)
    y_onehot: (num_classes, batch_num)
    Returns: scalar
    """
    y_smooth = label_smoothing(y_onehot, epsilon)
    probs = softmax_numpy(logits)
    return -np.mean(np.sum(y_smooth * np.log(probs + eps), axis=0))


# Compare hard vs smoothed targets
hard_target = np.array([[0], [0], [1], [0], [0]], dtype=float)
smooth_target = label_smoothing(hard_target, epsilon=0.1)

print("Hard target:    ", hard_target.ravel())
print("Smoothed (ε=0.1):", smooth_target.ravel())

# Loss comparison
test_logits = np.array([[1.0], [0.5], [3.0], [-1.0], [0.2]])
loss_hard = categorical_cross_entropy_numpy(test_logits, hard_target)
loss_smooth = cross_entropy_with_label_smoothing(test_logits, hard_target, epsilon=0.1)
print(f"\nCE (hard targets):     {loss_hard:.6f}")
print(f"CE (label smoothing):  {loss_smooth:.6f}")

# PyTorch equivalent
logits_pt_ls = torch.tensor(test_logits.T, dtype=torch.float32)
targets_pt_ls = torch.tensor([2], dtype=torch.long)
loss_pt_ls = F.cross_entropy(logits_pt_ls, targets_pt_ls, label_smoothing=0.1).item()
print(f"PyTorch CE (ε=0.1):    {loss_pt_ls:.6f}")

Hard target:     [0. 0. 1. 0. 0.]
Smoothed (ε=0.1): [0.02 0.02 0.92 0.02 0.02]

CE (hard targets):     0.259704
CE (label smoothing):  0.485704
PyTorch CE (ε=0.1):    0.485704


---
## 1.6 Comparison
| Loss | Task | Output activation | Key property |
|------|------|------------------|--------------|
| **BCE** | Binary classification | Sigmoid | MLE for Bernoulli; clean gradient $\hat{y}-y$ |
| **Categorical CE** | Multi-class | Softmax | MLE for categorical; equivalent to NLL(LogSoftmax) |
| **NLL + LogSoftmax** | Multi-class | LogSoftmax | Numerically stable decomposition of CE |
| **KL Divergence** | Distribution matching, VAEs | Depends | Measures divergence between distributions; CE - entropy |
| **Label Smoothing CE** | Multi-class (regularized) | Softmax | Prevents overconfidence; better calibration |

**Rules of thumb:**
- Binary task → BCE
- Multi-class → CE (PyTorch `cross_entropy` combines LogSoftmax + NLL)
- Want calibrated probabilities → label smoothing
- Knowledge distillation → KL divergence between teacher and student distributions
- Generative models (VAE) → reconstruction loss + KL divergence